# 1.0 Introduction

This notebook is used for Direct Preference Optimization of the earlier finetuned SFT model: 
- [robinsmits/Schaapje-2B-Chat-SFT-V1.0](https://huggingface.co/robinsmits/Schaapje-2B-Chat-SFT-V1.0).

The dataset [BramVanroy/ultra_feedback_dutch_cleaned](https://huggingface.co/datasets/BramVanroy/ultra_feedback_dutch_cleaned) is used for the DPO alignment.

## 1.1 Import Modules

In [1]:
# Import Modules
import os
import torch
from datasets import load_dataset
from peft import (AutoPeftModelForCausalLM,
                  LoraConfig, 
                  TaskType)
from transformers import (AutoTokenizer,
                          AutoModelForCausalLM)
from trl import (DPOConfig,
                 DPOTrainer)

## 1.2 Set Work Dir

In [ ]:
# Set Work Folder to use...
WORK_DIR = './SchaapjeTemp/'
os.makedirs(WORK_DIR, exist_ok = True)

## 1.3 Constants

In [ ]:
# Set Name Constants
sft_model_name = 'robinsmits/Schaapje-2B-Chat-SFT-V1.0'
dpo_model_name = 'Schaapje-2B-Chat-DPO-V1.0'
hf_organization_name = 'robinsmits'
hf_model_name = 'Schaapje-2B-Chat-V1.0'

## 1.4 Tokenizer

In [4]:
# Create Tokenizer
tokenizer = AutoTokenizer.from_pretrained(sft_model_name)
tokenizer.model_max_length = 4096
tokenizer.padding_side = 'left'
tokenizer.truncation_side = 'left'

# Tokenizer Summary
print(tokenizer)

GPT2TokenizerFast(name_or_path='robinsmits/Schaapje-2B-Chat-SFT-V1.0', vocab_size=49152, model_max_length=4096, is_fast=True, padding_side='left', truncation_side='left', special_tokens={'bos_token': '<|end_of_text|>', 'eos_token': '<|end_of_text|>', 'unk_token': '<|end_of_text|>', 'pad_token': '<|end_of_text|>', 'additional_special_tokens': ['<|start_of_role|>', '<|end_of_role|>', '<|tool_call|>']}, clean_up_tokenization_spaces=True, added_tokens_decoder={
	0: AddedToken("<|end_of_text|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<fim_prefix>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("<fim_middle>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("<fim_suffix>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	4: AddedToken("<fim_pad>", rstrip=False, lstrip=False, single_word=False, normalized=False

## 1.5 Create Model based on SFT Model

In [5]:
# Create Base Model
model = AutoModelForCausalLM.from_pretrained(sft_model_name,
                                             use_cache = False,
                                             torch_dtype = torch.bfloat16,
                                             device_map = 'auto')

# Show Model Summary
print(model)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

GraniteForCausalLM(
  (model): GraniteModel(
    (embed_tokens): Embedding(49155, 2048, padding_idx=0)
    (layers): ModuleList(
      (0-39): 40 x GraniteDecoderLayer(
        (self_attn): GraniteSdpaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): GraniteMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): GraniteRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): GraniteRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): GraniteRMSNorm((20

## 1.6 Set LoraConfig

In [6]:
# LoRA config
peftconfig = LoraConfig(r = 64,
                        lora_alpha = 16,
                        target_modules = 'all-linear',
                        lora_dropout = 0.05,
                        bias = 'none',
                        task_type = TaskType.CAUSAL_LM)

## 1.7 Load DPO Dataset

For DPO allignment the 'dpo_hq' subset is used as it is further cleaned. I also tried out the 'dpo_all' subset. I didn't really notice much difference when using the models that were trained on them.

However for safety and probably best allignment quality I will only use and publish the model based on the 'dpo_hq' subset.

In [7]:
# Load Dataset
dpo_dataset = load_dataset('BramVanroy/ultra_feedback_dutch_cleaned', 'dpo_hq')

# Split Datasets
dpo_train_dataset = dpo_dataset['train_prefs']
dpo_val_dataset = dpo_dataset['test_prefs']

# Summary
print(dpo_train_dataset)
print(dpo_val_dataset)

Dataset({
    features: ['prompt', 'chosen', 'rejected'],
    num_rows: 9186
})
Dataset({
    features: ['prompt', 'chosen', 'rejected'],
    num_rows: 484
})


In [8]:
# Used and modified from HuggingFace Alignment Handbook
def apply_chat_template(example, tokenizer):
    if all(k in example.keys() for k in ("chosen", "rejected")):
        # For DPO, the inputs are triples of (prompt, chosen, rejected), where `chosen` and `rejected` are the final turn of a dialogue
        # We therefore need to extract the N-1 turns to form the prompt
        prompt_messages = example["chosen"][:-1]

        # Now we extract the final turn to define chosen/rejected responses
        chosen_messages = example["chosen"][-1:]
        rejected_messages = example["rejected"][-1:]
        example["text_chosen"] = tokenizer.apply_chat_template(chosen_messages, tokenize=False)
        example["text_rejected"] = tokenizer.apply_chat_template(rejected_messages, tokenize=False)
        example["text_prompt"] = tokenizer.apply_chat_template(prompt_messages, tokenize=False)

    return example

In [9]:
# Get Original columns
original_columns = dpo_train_dataset.column_names

# Proces Train Dataset
dpo_train_dataset = dpo_train_dataset.map(apply_chat_template,
                                          fn_kwargs = {"tokenizer": tokenizer},
                                          remove_columns = original_columns)

# Proces Validation Dataset
dpo_val_dataset = dpo_val_dataset.map(apply_chat_template,
                                      fn_kwargs = {"tokenizer": tokenizer},
                                      remove_columns = original_columns)

# Rename columns
dpo_train_dataset = dpo_train_dataset.rename_columns({"text_prompt": "prompt", "text_chosen": "chosen", "text_rejected": "rejected"})
dpo_val_dataset = dpo_val_dataset.rename_columns({"text_prompt": "prompt", "text_chosen": "chosen", "text_rejected": "rejected"})

# Summary
print(dpo_train_dataset)
print(dpo_val_dataset)

Dataset({
    features: ['chosen', 'rejected', 'prompt'],
    num_rows: 9186
})
Dataset({
    features: ['chosen', 'rejected', 'prompt'],
    num_rows: 484
})


## 1.8 Train Model - Direct Preference Optimization

In [10]:
# Set Steps
eval_steps = 30
save_steps = 30
logging_steps = 30

# Set Learning Rate
learning_rate = 1.0e-5

# DPO Training Arguments
training_args = DPOConfig(num_train_epochs = 1,
                          learning_rate = learning_rate,
                          lr_scheduler_type = 'polynomial',
                          lr_scheduler_kwargs = {'power': 1, 'lr_end': 0.50 * learning_rate},
                          max_grad_norm = 1.0,
                          eval_strategy = "steps",
                          logging_steps = logging_steps,
                          save_strategy = 'steps',
                          eval_steps = eval_steps,
                          save_steps = save_steps,
                          save_total_limit = 3,
                          per_device_train_batch_size = 1,
                          per_device_eval_batch_size = 1,
                          gradient_accumulation_steps = 32,
                          gradient_checkpointing = True,
                          gradient_checkpointing_kwargs = {'use_reentrant': False},
                          warmup_ratio = 0.1,
                          bf16 = True,
                          beta = 0.1,
                          max_length = 2048,
                          max_prompt_length = 1536,
                          output_dir = f'{WORK_DIR}{hf_model_name}',
                          remove_unused_columns = False,
                          push_to_hub = False,
                          optim = 'adamw_bnb_8bit',
                          weight_decay = 0.001,
                          load_best_model_at_end = True,
                          report_to = 'none')

# Config DPOTrainer
dpo_trainer = DPOTrainer(model,
                         ref_model = None,
                         peft_config = peftconfig,    
                         args = training_args,
                         train_dataset = dpo_train_dataset,
                         eval_dataset = dpo_val_dataset,
                         processing_class = tokenizer)

# Train DPO Model
dpo_trainer.train()

# Save DPO Model
dpo_trainer.save_model()

Tokenizing train dataset:   0%|          | 0/9186 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/484 [00:00<?, ? examples/s]

  0%|          | 0/287 [00:00<?, ?it/s]

{'loss': 1.0019, 'grad_norm': 135.7120361328125, 'learning_rate': 9.980620155038761e-06, 'rewards/chosen': 0.037957679480314255, 'rewards/rejected': 0.01177685521543026, 'rewards/accuracies': 0.503125011920929, 'rewards/margins': 0.026180831715464592, 'logps/chosen': -979.986572265625, 'logps/rejected': -1066.879150390625, 'logits/chosen': -1.973187804222107, 'logits/rejected': -1.922747015953064, 'epoch': 0.1}


  0%|          | 0/484 [00:00<?, ?it/s]

...

{'eval_loss': 0.2787792682647705, 'eval_runtime': 411.3003, 'eval_samples_per_second': 1.177, 'eval_steps_per_second': 1.177, 'eval_rewards/chosen': -0.4129659831523895, 'eval_rewards/rejected': -3.0912599563598633, 'eval_rewards/accuracies': 0.913223147392273, 'eval_rewards/margins': 2.6782937049865723, 'eval_logps/chosen': -945.9951171875, 'eval_logps/rejected': -1065.901123046875, 'eval_logits/chosen': -2.0718514919281006, 'eval_logits/rejected': -2.0291996002197266, 'epoch': 0.73}
{'loss': 0.4383, 'grad_norm': 116.8138427734375, 'learning_rate': 5.910852713178295e-06, 'rewards/chosen': -0.5145475268363953, 'rewards/rejected': -3.245974540710449, 'rewards/accuracies': 0.8322916626930237, 'rewards/margins': 2.731426954269409, 'logps/chosen': -995.8994140625, 'logps/rejected': -1105.3104248046875, 'logits/chosen': -2.0763838291168213, 'logits/rejected': -2.0315425395965576, 'epoch': 0.84}


  0%|          | 0/484 [00:00<?, ?it/s]

{'eval_loss': 0.27952733635902405, 'eval_runtime': 411.2857, 'eval_samples_per_second': 1.177, 'eval_steps_per_second': 1.177, 'eval_rewards/chosen': -0.5359205603599548, 'eval_rewards/rejected': -3.406750440597534, 'eval_rewards/accuracies': 0.913223147392273, 'eval_rewards/margins': 2.8708298206329346, 'eval_logps/chosen': -947.224609375, 'eval_logps/rejected': -1069.0560302734375, 'eval_logits/chosen': -2.075483798980713, 'eval_logits/rejected': -2.0335631370544434, 'epoch': 0.84}
{'loss': 0.448, 'grad_norm': 93.66802978515625, 'learning_rate': 5.329457364341086e-06, 'rewards/chosen': -0.38568297028541565, 'rewards/rejected': -3.0930747985839844, 'rewards/accuracies': 0.8374999761581421, 'rewards/margins': 2.7073919773101807, 'logps/chosen': -986.2472534179688, 'logps/rejected': -1077.3963623046875, 'logits/chosen': -2.08368182182312, 'logits/rejected': -2.03359317779541, 'epoch': 0.94}


  0%|          | 0/484 [00:00<?, ?it/s]

{'eval_loss': 0.27413618564605713, 'eval_runtime': 411.2354, 'eval_samples_per_second': 1.177, 'eval_steps_per_second': 1.177, 'eval_rewards/chosen': -0.3499433696269989, 'eval_rewards/rejected': -3.2382097244262695, 'eval_rewards/accuracies': 0.9173553586006165, 'eval_rewards/margins': 2.888266086578369, 'eval_logps/chosen': -945.3648071289062, 'eval_logps/rejected': -1067.37060546875, 'eval_logits/chosen': -2.0749032497406006, 'eval_logits/rejected': -2.0324201583862305, 'epoch': 0.94}
{'train_runtime': 24114.0081, 'train_samples_per_second': 0.381, 'train_steps_per_second': 0.012, 'train_loss': 0.5936453616577574, 'epoch': 1.0}


## 1.9 Push to HF Hub

In [11]:
# Memory Cleanup
del model, dpo_trainer
torch.cuda.empty_cache()

# Reload Model
model = AutoPeftModelForCausalLM.from_pretrained(training_args.output_dir,
                                                 device_map = 'auto',
                                                 torch_dtype = torch.bfloat16)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [12]:
# Model
model = model.merge_and_unload(safe_merge = True)
model.push_to_hub(f'{hf_organization_name}/{hf_model_name}', private = True, commit_message = 'Final Chat Model')

# Tokenizer
tokenizer.push_to_hub(f'{hf_organization_name}/{hf_model_name}', private = True, commit_message = 'Final Chat Model')

Upload 2 LFS files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/67.1M [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/robinsmits/Schaapje-2B-Chat-V1.0/commit/635a40ae2c0ca8b334bd60e0517d93751fe62692', commit_message='Final Chat Model', commit_description='', oid='635a40ae2c0ca8b334bd60e0517d93751fe62692', pr_url=None, repo_url=RepoUrl('https://huggingface.co/robinsmits/Schaapje-2B-Chat-V1.0', endpoint='https://huggingface.co', repo_type='model', repo_id='robinsmits/Schaapje-2B-Chat-V1.0'), pr_revision=None, pr_num=None)